In [1]:
#Sheet Pvt

In [2]:
import pandas as pd

df = pd.read_excel('PTC draft.xlsx')


In [3]:
df["Year_Month"] = (
    df["Year Reqrd."].astype(str) + "-" +
    df["Mon Reqrd."].astype(str).str.zfill(2)
)


In [4]:
pivot_df = (
    df.pivot_table(
        index=["PN AL78", "PN used", "Description", "Seqnc", "Qty Resvered"],
        columns="Year_Month",
        values="Qty Open",
        aggfunc="sum",
        fill_value=""
    )
    .reset_index()
)


In [5]:
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str)

pivot_df = (
    pivot_df
    .sort_values(by=["PN AL78", "Seqnc"], ascending=[True, True])
    .reset_index(drop=True)
)


In [6]:
pivot_df.insert(
    loc=0,
    column="PN",
    value=pivot_df["PN AL78"]
)

# === CONFIG ===
current_period = pd.Period("2025-12", freq="M")

# === PREP RAW DATA ===
df["Period"] = pd.PeriodIndex(
    df["Year Reqrd."].astype(str) + "-" +
    df["Mon Reqrd."].astype(str).str.zfill(2),
    freq="M"
)

# === FILTER 7 MONTHS PRIOR ===
prior_7m_df = df[
    (df["Period"] < current_period) &
    (df["Period"] >= current_period - 7)
]

# === ROW-LEVEL AGGREGATION (PN + Seqnc) ===
prior_7m_sum = (
    prior_7m_df
    .groupby(["PN AL78", "Seqnc"], as_index=False)["Qty Open"]
    .sum()
    .rename(columns={"Qty Open": "7 Months Prior"})
)

# === MERGE INTO PIVOT TABLE ===
pivot_df = pivot_df.merge(
    prior_7m_sum,
    on=["PN AL78", "Seqnc"],
    how="left"
)

pivot_df["7 Months Prior"] = pivot_df["7 Months Prior"].fillna(0).astype(int)


In [7]:
# Take current column order
cols = pivot_df.columns.tolist()

# Remove PN from its current position
cols.remove("PN")

# Insert PN right after "03" (2026-03)
insert_pos = cols.index("2026-03") + 1
cols.insert(insert_pos, "PN")

# Reorder dataframe
pivot_df = pivot_df[cols]


In [8]:
# =========================
# OPEN ORDERS — FINAL FIX
# =========================
#TIAP TARIK DATA UBAH2 INI MONTH_COLS NYA
month_cols = [
    "2025-06","2025-07","2025-08","2025-09","2025-10",
    "2025-11","2025-12","2026-01",
    "2026-02","2026-03","2026-04"
]

# Ensure numeric
pivot_df[month_cols] = (
    pivot_df[month_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)
#INI JUGA
# Base month = Jan 2026
base_idx = month_cols.index("2026-01")

pivot_df["OO CM"]  = pivot_df[month_cols[base_idx]] + pivot_df[month_cols[base_idx + 1]]
pivot_df["OO NM"]  = pivot_df[month_cols[base_idx + 2]]
pivot_df["OO N2M"] = pivot_df[month_cols[base_idx + 3]]
pivot_df["OO N3M"] = 0 

# Ensure int
pivot_df[["OO CM","OO NM","OO N2M","OO N3M"]] = (
    pivot_df[["OO CM","OO NM","OO N2M","OO N3M"]].astype(int)
)

# === PLACE AFTER '7 Months Prior' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("7 Months Prior") + 1

for col in ["OO CM","OO NM","OO N2M","OO N3M"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [9]:
import pandas as pd
import numpy as np

# === READ FILES ===
pncheck_df = pd.read_excel("Pvt chkPN draft.xlsx", dtype=str)
FCPTC_df = pd.read_excel(
    "Forecast OvH PTC Dec2025.xlsx",
    sheet_name="Pvt",
    skiprows=2
)

# === NORMALIZE PN FORMAT ===
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN AL78"] = pncheck_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN Final"] = pncheck_df["PN Final"].astype(str).str.strip()

# =========================================================
# 1️⃣ ADD PN.Final (lookup PN AL78 → PN Final)
# =========================================================
pivot_df = pivot_df.merge(
    pncheck_df[["PN AL78", "PN Final"]]
        .rename(columns={"PN Final": "PN.Final"}),
    on="PN AL78",
    how="left"
)

# === PLACE 'PN.Final' AFTER 'OO N3M' ===
cols = pivot_df.columns.tolist()
cols.remove("PN.Final")
cols.insert(cols.index("OO N3M") + 1, "PN.Final")
pivot_df = pivot_df[cols]

# =========================================================
# 2️⃣ FC LOOKUP USING *PN.Final*
# =========================================================
fc_lookup = (
    FCPTC_df[["PART NO", " M2", " M3", " M4", " M5"]]
    .rename(columns={
        "PART NO": "PN.Final",
        " M2": "FC CM",
        " M3": "FC NM",
        " M4": "FC N2M",
        " M5": "FC N3M"
    })
)

pivot_df = pivot_df.merge(
    fc_lookup,
    on="PN.Final",
    how="left"
)

# === CLEAN FC VALUES (#N/A → 0, remove .0) ===
for col in ["FC CM", "FC NM", "FC N2M", "FC N3M"]:
    pivot_df[col] = (
        pd.to_numeric(pivot_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

# === ADD FC N4M (BLANK) ===
pivot_df["FC N4M"] = ""

# === PLACE FC COLUMNS AFTER 'PN.Final' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("PN.Final") + 1

for col in ["FC CM", "FC NM", "FC N2M", "FC N3M", "FC N4M"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [10]:
# === ADD 2 EMPTY COLUMNS ===
pivot_df[""] = ""
pivot_df["  "] = ""

# === ADD LONG PN ===
pivot_df["Long PN"] = pivot_df["PN used"]

# === MOVE COLUMNS TO THE RIGHT OF 'FC N4M' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("FC N4M") + 1

for col in ["", "  ", "Long PN"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [11]:
import numpy as np

# === ENSURE NUMERIC (SAFETY) ===
num_cols = [
    "FC CM", "OO CM",
    "FC NM", "OO NM",
    "FC N2M", "OO N2M",
    "FC N3M", "OO N3M"
]

for col in num_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === CALCULATIONS (ROW-LEVEL, EXCEL-EQUIVALENT) ===
pivot_df["Curr. Month"] = (
    pivot_df[["FC CM", "OO CM"]].max(axis=1) +
    pivot_df[["FC NM", "OO NM"]].max(axis=1)
)

pivot_df["Req CM"] = pivot_df[["FC N2M", "OO N2M"]].max(axis=1)

pivot_df["Req NM"] = pivot_df[["FC N3M", "OO N3M"]].max(axis=1)

# === BLANK COLUMNS ===
pivot_df["Req N2M"] = ""
pivot_df["Req N3M"] = ""

# === PLACE COLUMNS AFTER 'Long PN' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("Long PN") + 1

for col in ["Curr. Month", "Req CM", "Req NM", "Req N2M", "Req N3M"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [12]:
# === READ PARTS MOVEMENT DATA ===
partsmovementPTC_df = pd.read_excel(
    "pmovdcE 19 DR Jan2026.xlsx",
    sheet_name="OH All"
)

# === NORMALIZE PN KEYS (CRITICAL) ===
partsmovementPTC_df["Last PN."] = (
    partsmovementPTC_df["Last PN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

pivot_df["PN.Final"] = (
    pivot_df["PN.Final"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# === AGGREGATE OH PER Last PN.1 ===
oh_lookup = (
    partsmovementPTC_df
    .groupby("Last PN", as_index=False)["OH All"]
    .sum()
    .rename(columns={
        "Last PN": "PN.Final",
        "OH All": "OH"
    })
)

# === MERGE USING PN.Final ===
pivot_df = pivot_df.merge(
    oh_lookup,
    on="PN.Final",
    how="left"
)

# === ENSURE NUMERIC (KEEP NaN) ===
pivot_df["OH"] = pd.to_numeric(
    pivot_df["OH"],
    errors="coerce"
)

# === PLACE COLUMN AFTER 'Req N3M' ===
cols = pivot_df.columns.tolist()
cols.remove("OH")
cols.insert(cols.index("Req N3M") + 1, "OH")

pivot_df = pivot_df[cols]


In [ ]:
# =========================
# RECEIVING AND TRANSIT
# =========================

# === READ DATA ===
osgrr_df = pd.read_excel(
    "Outstd.GRR PTC 17Jan2026.xlsx",
    sheet_name="Pvt",
    skiprows=3
)

shipment_df = pd.read_excel(
    "Shipment PRP.xlsx",
    sheet_name="noGRR",
    skiprows=2
)

# === AGGREGATE SI (by Last PN) ===
rcv_si = (
    osgrr_df
    .groupby("Last PN", as_index=False)["InRcv.1"]
    .sum()
    .rename(columns={"Last PN": "PN.Final", "InRcv.1": "In Rcv SI"})
)

tr_si = (
    osgrr_df
    .groupby("Last PN", as_index=False)["InTr.1"]
    .sum()
    .rename(columns={"Last PN": "PN.Final", "InTr.1": "In Tr SI"})
)

# === AGGREGATE NON-SI (by PN Final) ===
rcv = (
    shipment_df
    .groupby("PN Final", as_index=False)["In\nRcv.1"]
    .sum()
    .rename(columns={"PN Final": "PN.Final", "In\nRcv.1": "In Rcv"})
)

tr = (
    shipment_df
    .groupby("PN Final", as_index=False)["In\nTr.1"]
    .sum()
    .rename(columns={"PN Final": "PN.Final", "In\nTr.1": "In Tr"})
)

# === MERGE ALL LOOKUPS USING PN.Final ===
for df_add in [rcv_si, tr_si, rcv, tr]:
    pivot_df = pivot_df.merge(
        df_add,
        on="PN.Final",
        how="left"
    )

# === CLEAN NaN → 0 ===
for col in ["In Rcv SI", "In Tr SI", "In Rcv", "In Tr"]:
    pivot_df[col] = (
        pd.to_numeric(pivot_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

# === TOTAL COLUMNS ===
pivot_df["In Rcv Total"] = pivot_df["In Rcv SI"] + pivot_df["In Rcv"]
pivot_df["In Tr Total"]  = pivot_df["In Tr SI"]  + pivot_df["In Tr"]

# === PLACE COLUMNS BESIDE 'OH' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("OH") + 1

for col in [
    "In Rcv Total", "In Tr Total",
    "In Rcv SI", "In Tr SI",
    "In Rcv", "In Tr"
]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [ ]:
import numpy as np

# === ENSURE NUMERIC INPUTS ===
check_cols = [
    "OH",
    "Curr. Month",
    "In Rcv Total",
    "In Tr Total",
    "7 Months Prior",
    "OO CM",
    "OO NM"
]

for col in check_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === chk OH to CM LOGIC (EXCEL-EQUIVALENT) ===
pivot_df["chk OH to CM"] = np.where(
    pivot_df["OH"] < pivot_df["Curr. Month"],
    np.where(
        pivot_df["Curr. Month"] <= (
            pivot_df["OH"]
            + pivot_df["In Rcv Total"]
            + pivot_df["In Tr Total"]
            + pivot_df["7 Months Prior"]
            + pivot_df["OO CM"]
            + pivot_df["OO NM"]
        ),
        "OK",
        "NG"
    ),
    "OK"
)

# === PLACE COLUMN AFTER 'In Tr Total' ===
cols = pivot_df.columns.tolist()
cols.remove("chk OH to CM")
cols.insert(cols.index("In Tr") + 1, "chk OH to CM")

pivot_df = pivot_df[cols]


In [ ]:
# === ENSURE NUMERIC INPUTS ===
nm_cols = [
    "OH",
    "In Rcv Total",
    "In Tr Total",
    "Curr. Month",
    "Req CM"
]

for col in nm_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === chk OH to NM (EXCEL-EQUIVALENT) ===
pivot_df["chk OH to NM"] = (
    pivot_df["OH"]
    + pivot_df["In Rcv Total"]
    + pivot_df["In Tr Total"]
    - pivot_df["Curr. Month"]
    - pivot_df["Req CM"]
)

# === PLACE COLUMN AFTER 'chk OH to CM' ===
cols = pivot_df.columns.tolist()
cols.remove("chk OH to NM")
cols.insert(cols.index("chk OH to CM") + 1, "chk OH to NM")

pivot_df = pivot_df[cols]


In [ ]:
# === ENSURE NUMERIC INPUTS ===
req_cols = ["Curr. Month", "Req CM"]

for col in req_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === Req CM Total (EXCEL-EQUIVALENT) ===
pivot_df["Req CM Total"] = (
    pivot_df["Curr. Month"] + pivot_df["Req CM"]
)

# === PLACE COLUMN AFTER 'chk OH to NM' ===
cols = pivot_df.columns.tolist()
cols.remove("Req CM Total")
cols.insert(cols.index("chk OH to NM") + 1, "Req CM Total")

pivot_df = pivot_df[cols]


In [ ]:
# === ENSURE NUMERIC INPUTS ===
opor_cols = ["7 Months Prior", "OO CM"]

for col in opor_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === OpOrd CM (EXCEL-EQUIVALENT) ===
pivot_df["OpOrd CM"] = (
    pivot_df["7 Months Prior"] + pivot_df["OO CM"]
)

# === PLACE COLUMN AFTER 'Req CM Total' ===
cols = pivot_df.columns.tolist()
cols.remove("OpOrd CM")
cols.insert(cols.index("Req CM Total") + 1, "OpOrd CM")

pivot_df = pivot_df[cols]


In [ ]:
# === ADD DESCRIPTION 2 ===
pivot_df["Description 2"] = pivot_df["Description"]

# === PLACE COLUMN AFTER 'OpORd CM' ===
cols = pivot_df.columns.tolist()
cols.remove("Description 2")
cols.insert(cols.index("OpOrd CM") + 1, "Description 2")

pivot_df = pivot_df[cols]


In [ ]:
# === READ SOURCE FILE ===
ptc_src_df = pd.read_excel("PTC draft.xlsx")

# === BUILD LOOKUP KEY IN SOURCE ===
ptc_src_df["lookup_key"] = (
    ptc_src_df["Seqnc"].astype(str) + "." +
    ptc_src_df["PN AL78"].astype(str)
)

esd_lookup = (
    ptc_src_df[["lookup_key", "ESD Mon"]]
    .drop_duplicates()
)

# === BUILD LOOKUP KEY IN PIVOT_DF ===
pivot_df["lookup_key"] = (
    pivot_df["Seqnc"].astype(str) + "." +
    pivot_df["PN AL78"].astype(str)
)

# === MERGE (VLOOKUP STYLE) ===
pivot_df = pivot_df.merge(
    esd_lookup,
    on="lookup_key",
    how="left"
)

# === CLEAN UP ===
pivot_df["Mon. ESD"] = pivot_df["ESD Mon"].fillna("")

pivot_df.drop(columns=["lookup_key", "ESD Mon"], inplace=True)

# === PLACE COLUMN AFTER 'Description 2' ===
cols = pivot_df.columns.tolist()
cols.remove("Mon. ESD")
cols.insert(cols.index("Description 2") + 1, "Mon. ESD")

pivot_df = pivot_df[cols]
# === CONVERT Mon. ESD TO INTEGER (NO .0, BLANK-SAFE) ===
pivot_df["Mon. ESD"] = (
    pd.to_numeric(pivot_df["Mon. ESD"], errors="coerce")
    .astype("Int64")   # nullable integer
)


In [ ]:
# === ADD Qty Reserved (COPY OF Qty Resvrd.) ===
pivot_df["Qty Reserved"] = pivot_df["Qty Resvered"]

# === PLACE COLUMN AFTER 'Mon. ESD' ===
cols = pivot_df.columns.tolist()
cols.remove("Qty Reserved")
cols.insert(cols.index("Mon. ESD") + 1, "Qty Reserved")

pivot_df = pivot_df[cols]


In [ ]:
# === READ PN CHECK FILE ===
pncheck_df = pd.read_excel("Pvt chkPN draft.xlsx", dtype=str)

# === FORCE STRING (IMPORTANT) ===
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN AL78"] = pncheck_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN Final"] = pncheck_df["PN Final"].astype(str).str.strip()

# === MERGE: PN AL78 → PN Final ===
pivot_df = pivot_df.merge(
    pncheck_df[["PN AL78", "PN Final"]].drop_duplicates(),
    on="PN AL78",
    how="left"
)



In [ ]:
# === ADD EMPTY COLUMN AFTER 'PN Final' ===
pivot_df["   "] = ""

# === ADD chk PN COLUMN ===
pivot_df["chk PN"] = np.where(
    pivot_df["PN.Final"].astype(str) == pivot_df["PN AL78"].astype(str),
    "OK",
    "NG"
)

# === PLACE COLUMNS AFTER 'PN Final' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("PN.Final") + 1

for col in ["   ", "chk PN"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [ ]:
pivot_df.to_excel("Pvt draft.xlsx", index=False)